In [ ]:
# Colab 請選 2026.07；安裝到獨立環境，避免更改核心已載入的 NumPy。
from pathlib import Path
import subprocess
import sys
import tempfile

assert sys.version_info[:3] == (3, 12, 13), "請使用書籍或 Colab 2026.07 的 Python 3.12.13"
python = sys.executable
if "google.colab" in sys.modules:
    %pip -q install virtualenv
    env = Path("/content/ch08-book-env")
    if not (env / "bin/python").is_file():
        subprocess.run([python, "-m", "virtualenv", "-q", str(env)], check=True)
    python = str(env / "bin/python")
    pins = ["deepchem==2.8.1.dev20260821122258", "numpy==1.26.4", "scipy==1.17.1",
            "rdkit==2026.9.1", "scikit-learn==1.9.1", "pandas==3.0.6", "matplotlib==3.11.2"]
    subprocess.run([python, "-m", "pip", "install", "-q", *pins], check=True)
    subprocess.run([python, "-m", "pip", "check"], check=True)
work = tempfile.TemporaryDirectory()
script = Path(work.name) / "08-04-hdbscan.py"

## ESOL 的密度分群
從 DeepChem 載入公開結構，以分子量、計算 log P 與 TPSA 分群。下格產生完整程式，最後一格才執行。初次下載資料需要連網。

In [ ]:
%%writefile "{script}"
from pathlib import Path
import os
import tempfile

import deepchem as dc
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors, Draw
from sklearn.cluster import HDBSCAN
from sklearn.preprocessing import StandardScaler

# 固定資料順序與版本；此處分群沒有隨機初始化。
np.random.seed(42)
cache = Path(os.environ.get("DEEPCHEM_DATA_DIR", Path(tempfile.gettempdir()) / "cheminfo-datasets"))
cache.mkdir(parents=True, exist_ok=True)
_, splits, _ = dc.molnet.load_delaney(
    featurizer=dc.feat.RawFeaturizer(smiles=True), splitter="scaffold",
    transformers=[], data_dir=str(cache), reload=False)
records = [str(s).strip() for split in splits for s in split.ids]
keys = sorted({Chem.MolToSmiles(Chem.MolFromSmiles(s)) for s in records})
mols = [Chem.MolFromSmiles(s) for s in keys]
values = np.array([[Descriptors.MolWt(m), Descriptors.MolLogP(m),
                    Descriptors.TPSA(m)] for m in mols])
assert len(records) == 1128 and np.isfinite(values).all()
print("ESOL records, unique structures:", len(records), len(keys))

Z = StandardScaler().fit_transform(values)
def group(min_size):
    return HDBSCAN(min_cluster_size=min_size, min_samples=6, metric="euclidean",
                   algorithm="brute", copy=True).fit_predict(Z)

labels = group(20)
coarser = group(30)
for minimum, result in [(20, labels), (30, coarser)]:
    print("min_cluster_size =", minimum, "label counts:",
          dict(zip(*np.unique(result, return_counts=True))))
print("Rows are baseline groups; columns give new label counts")
for label in np.unique(labels):
    print(label, dict(zip(*np.unique(coarser[labels == label], return_counts=True))))

examples = [("Benzene", "c1ccccc1"), ("Cyclohexane", "C1CCCCC1"),
            ("Tetrabromomethane", "BrC(Br)(Br)Br")]
print("Structure | cluster | MW | MolLogP | TPSA | sixth-neighbor distance (including self)")
fig = plt.figure(figsize=(10, 5.2), layout="constrained")
grid = fig.add_gridspec(3, 2, width_ratios=[2.1, 1])
ax = fig.add_subplot(grid[:, 0])
colors = {-1: "#bfc2c6", 0: "#a04b15", 1: "#4d80b2", 2: "#cc6c9c", 3: "#188476", 4: "#6952a0"}
for label in sorted(set(labels)):
    mask = labels == label
    color = colors.get(label, "#47676c")
    name = "Unassigned" if label == -1 else f"Group {label}"
    ax.scatter(*values[mask, :2].T, s=12, color=color, alpha=.68, linewidths=0,
               label=f"{name} ({sum(mask)})")
ax.set_xlabel("Molar mass (g/mol)")
ax.set_ylabel("Calculated log P")
ax.set_title("Two of the three clustering features", loc="left")
ax.legend(fontsize=8, frameon=False, loc="lower right", ncol=2)
for row, (name, smiles) in enumerate(examples):
    i = keys.index(Chem.MolToSmiles(Chem.MolFromSmiles(smiles)))
    mw, logp, tpsa = values[i]
    d6 = np.sort(np.linalg.norm(Z - Z[i], axis=1))[5]
    print(name, labels[i], np.round(values[i], 3), round(d6, 3))
    panel = fig.add_subplot(grid[row, 1])
    panel.imshow(Draw.MolToImage(mols[i], size=(280, 134)))
    panel.set_title(f"{name}  |  " + ("Unassigned" if labels[i] == -1 else f"Group {labels[i]}"),
                    fontsize=10, color="#202a37")
    panel.set_xlabel(f"MW {mw:.1f}   log P {logp:.2f}   TPSA {tpsa:.1f}", fontsize=8)
    panel.set_xticks([])
    panel.set_yticks([])
    for spine in panel.spines.values():
        spine.set_visible(False)
figure_dir = Path("../../figures") if Path("../../outline.yml").is_file() else Path("figures")
figure_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(figure_dir / "08-04-hdbscan.png", dpi=160)


In [ ]:
from IPython.display import Image, display

subprocess.run([python, str(script)], check=True)
figure_dir = Path("../../figures") if Path("../../outline.yml").is_file() else Path("figures")
display(Image(filename=str(figure_dir / "08-04-hdbscan.png")))